# Q1: Demonstrate the Roles of Driver, Cluster Manager, and Executor in a Spark Application

## Objective
Create a simple Spark DataFrame and perform a transformation and an action to understand how the Driver, Cluster Manager, and Executors work together during execution.

## Explanation

In this practical example:

- The **Driver** starts the Spark application, creates the SparkSession, and plans the execution.
- The **Cluster Manager** allocates resources (Executors) required to run the application.
- The **Executors** execute the assigned tasks, process the data, and return the results to the Driver.

The code below demonstrates this execution flow.

In [0]:

data = [
    (1, "Laptop", 65000),
    (2, "Mobile", 30000),
    (3, "Tablet", 25000),
    (4, "Monitor", 18000)
]
df = spark.createDataFrame(data, ["product_id", "product_name", "price"])

filtered_df = df.filter(df.price > 25000)
filtered_df.show()

+----------+------------+-----+
|product_id|product_name|price|
+----------+------------+-----+
|         1|      Laptop|65000|
|         2|      Mobile|30000|
+----------+------------+-----+



## Expected Output

| product_id | product_name | price |
|------------|--------------|-------|
| 1 | Laptop | 65000 |
| 2 | Mobile | 30000 |

## Code Explanation

### Step 1
The Driver creates a Spark DataFrame from the given data.

### Step 2
The `filter()` operation creates a new DataFrame containing products whose price is greater than 25,000.

This is a **Transformation**, so Spark does not execute it immediately.

### Step 3
The `show()` function is an **Action**.

When `show()` is executed:

- The Driver creates an execution plan (DAG).
- The Driver requests resources from the Cluster Manager.
- The Cluster Manager assigns available Executors.
- Executors process the filtered data in parallel.
- The processed result is sent back to the Driver.
- The Driver displays the output.

## Execution Flow

Driver
↓
Creates Execution Plan (DAG)
↓
Cluster Manager
↓
Allocates Executors
↓
Executors Execute Tasks
↓
Results Returned to Driver
↓
show() Displays Output

## Key Takeaway

This example demonstrates that the Driver coordinates the application, the Cluster Manager allocates resources, and the Executors perform the actual data processing.

## Q2: Demonstrate Spark's Lazy Evaluation


In [0]:
data = [
    (1, "Laptop", 65000),
    (2, "Mobile", 30000),
    (3, "Tablet", 25000),
    (4, "Monitor", 18000)
]

df = spark.createDataFrame(data, ["product_id", "product_name", "price"])
result_df = (
    df.filter(df.price > 20000)
      .select("product_name", "price")
      .withColumn("discount_price", df.price * 0.9)
)
result_df.show()

+------------+-----+--------------+
|product_name|price|discount_price|
+------------+-----+--------------+
|      Laptop|65000|       58500.0|
|      Mobile|30000|       27000.0|
|      Tablet|25000|       22500.0|
+------------+-----+--------------+



## Code Explanation

- `filter()`, `select()`, and `withColumn()` are **Transformations**.
- Spark stores these operations but does **not** execute them immediately.
- `show()` is an **Action** that triggers the execution of all transformations together.
- Spark builds an optimized **DAG** and executes it efficiently.


Spark's Lazy Evaluation improves performance by delaying execution, optimizing the execution plan, and processing the data only when an action is invoked.

## Q3: Write a Spark command to read a CSV file located at "data/source.csv", ensuring the first row is treated as a header and inferSchema is enabled. 

In [0]:
# Read CSV file with header and schema inference
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Volumes/practice_catalog/new_source/new_volumeforcei/Sales.csv")

df.limit(5).display()

Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales
FDA15,9.3,Low Fat,0.016047301,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.138
DRC01,5.92,Regular,0.019278216,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2,443.4228
FDN15,17.5,Low Fat,0.016760075,Meat,141.618,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.27
FDX07,19.2,Regular,0.0,Fruits and Vegetables,182.095,OUT010,1998,null,Tier 3,Grocery Store,732.38
NCD19,8.93,Low Fat,0.0,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052


In [0]:
df.printSchema()

root
 |-- Item_Identifier: string (nullable = true)
 |-- Item_Weight: double (nullable = true)
 |-- Item_Fat_Content: string (nullable = true)
 |-- Item_Visibility: double (nullable = true)
 |-- Item_Type: string (nullable = true)
 |-- Item_MRP: double (nullable = true)
 |-- Outlet_Identifier: string (nullable = true)
 |-- Outlet_Establishment_Year: integer (nullable = true)
 |-- Outlet_Size: string (nullable = true)
 |-- Outlet_Location_Type: string (nullable = true)
 |-- Outlet_Type: string (nullable = true)
 |-- Item_Outlet_Sales: double (nullable = true)



### Code Explanation
- `spark.read.format("csv")` specifies the file format.
- `.option("header", "true")` reads the first row as column names.
- `.option("inferSchema", "true")` automatically detects data types.
- `.load()` loads the CSV file into a DataFrame.
- `display()` displays the records.
- `printSchema()` displays the inferred schema.

Q4: What is the difference between CSV and Parquet in terms of storage (row-based vs. columnar) and why does it matter for performance? 

In [0]:
csv_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Volumes/practice_catalog/new_source/new_volumeforcei/Sales.csv")

csv_df.write.mode("overwrite").parquet("/Volumes/practice_catalog/new_source/new_volumeforcei/Sales_Parquet")
parquet_df = spark.read.parquet("/Volumes/practice_catalog/new_source/new_volumeforcei/Sales_Parquet")

print("CSV Data")
csv_df.show()

CSV Data
+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|
+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+
|          FDA15|        9.3|         Low Fat|    0.016047301|               Dairy|249.8092|           OUT049|                     1999|     Medium|              Tier 1|Supermarket Type1|         3735.138|
|          DRC01|       5.92|         Regular|    0.019278216|         Soft Drinks| 48.2692|           OUT018|                     2009|     Medium|              Tier 

In [0]:
print("Parquet Data")
parquet_df.limit(5).display()

Parquet Data


Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales
FDA15,9.3,Low Fat,0.016047301,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.138
DRC01,5.92,Regular,0.019278216,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2,443.4228
FDN15,17.5,Low Fat,0.016760075,Meat,141.618,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.27
FDX07,19.2,Regular,0.0,Fruits and Vegetables,182.095,OUT010,1998,null,Tier 3,Grocery Store,732.38
NCD19,8.93,Low Fat,0.0,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052
